## Notebook14a

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds-py/refs/heads/main/funs.py

In [ ]:
import numpy as np
import polars as pl

from funs import *
from plotnine import *
from polars import col as c

import torch.nn as nn
import torch.optim as optim

theme_set(theme_minimal())
pl.Config(tbl_rows=25)

ub = "https://raw.githubusercontent.com/taylor-arnold/fds-py-nb/refs/heads/main/"

In [ ]:
!wget -nc https://humanitiesdata.org/media/fashionmnist_10000.tar -P media/
!tar -xf media/fashionmnist_10000.tar -C media/
!wget -nc https://humanitiesdata.org/media/emnist_10000.tar -P media/
!tar -xf media/emnist_10000.tar -C media/

In [ ]:
fmnist = pl.read_csv(ub + "data/fashionmnist_10000.csv")
emnist = pl.read_csv(ub + "data/emnist_10000.csv")

### Part I: Fashion MNIST

1. Use the code below to create the training and testing data. Notice this time that the shape of the data has changed. Make sure you understand what this means.

In [ ]:
X, X_train, X_test, y, y_train, y_test, cn = DSTorch.load_image(
    fmnist, scale=True
)
X.shape

2. Next, use the following code to build a CNN model. Go through the parameters to understand the shape of the model.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.MaxPool2d(2),
          
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

3. As before, we will use the Adam optimizer. After a little trial and error for the above parameters, I found a slightly higher learning rate (0.003) compared to last time was ideal.

In [ ]:
model = SimpleCNN()
optimizer = optim.Adam(model.parameters(), lr=0.003)

4. Train the model. Notice that that the training loss will be worse because of the dropout, but it should help avoid overfitting.

In [ ]:
DSTorch.train(
    model, optimizer, X_train, y_train, num_epochs=20, batch_size=64
)

5. Look at the scores and confusion matrix of the training and testing sets. You will probably notice that the model does a bit better than the dense network, but the biggest improvement here is the dropout stopping the model from overfitting the training data.

In [ ]:
DSTorch.score_image(model, X_train, y_train, cn)

In [ ]:
DSTorch.score_image(model, X_test, y_test, cn)

In [ ]:
DSTorch.confusion_matrix(model, X_test, y_test, cn)

6. Finally, let's look at the negative examples. How easily can *you* tell them apart?

In [ ]:
(
    fmnist
    .with_columns(
        DSTorch.predict(model, X, y, cn)
    )
    .filter(c.target_ != c.prediction_)
    .with_columns(
        info = pl.concat_str(c.target_, c.prediction_, separator="=>")
    )
    .pipe(DSImage.plot_image_grid, label_name="info")
)

### Part II: EMNIST

7. Now look at the EMNIST dataset. We mostly following the same instructions, but notice where things change a bit.

In [ ]:
X, X_train, X_test, y, y_train, y_test, cn = DSTorch.load_image(
    emnist, scale=True
)
X.shape

8. Next, use the following code to build a CNN model. Go through the parameters to understand the shape of the model.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.MaxPool2d(2),
          
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

9. As before, we will use the Adam optimizer. After a little trial and error for the above parameters, I found a slightly higher learning rate (0.003) compared to last time was ideal.

In [ ]:
model = SimpleCNN(num_classes=len(cn))
optimizer = optim.Adam(model.parameters(), lr=0.003)

10. Train the model. Notice that that the training loss will be worse because of the dropout, but it should help avoid overfitting.

In [ ]:
DSTorch.train(
    model, optimizer, X_train, y_train, num_epochs=20, batch_size=64
)

11. Look at the scores of the training and testing sets. You will probably notice that the model does a bit better than the dense network, but the biggest improvement here is the dropout stopping the model from overfitting the training data.

In [ ]:
DSTorch.score_image(model, X_train, y_train, cn)

In [ ]:
DSTorch.score_image(model, X_test, y_test, cn)

12. Finally, let's look at the negative examples. How easily can *you* tell them apart?

In [ ]:
(
    emnist
    .with_columns(
        DSTorch.predict(model, X, y, cn)
    )
    .filter(c.target_ != c.prediction_)
    .with_columns(
        info = pl.concat_str(c.target_, c.prediction_, separator="=>")
    )
    .pipe(DSImage.plot_image_grid, label_name="info")
)